[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-13-cards-notifications.ipynb#scrollTo=a1b2c3d4)

---
# Day 13 · Flow Cards and Observability
**certified-journeys / metaflow-certified** · Day 13 · Review

> **Goal for today:** By the end of this notebook you can generate automatic run reports with `@card`, build custom cards with tables and charts, and articulate a concise mental model of every Metaflow decorator covered so far.


In [ ]:
%pip install -q metaflow scikit-learn pandas numpy matplotlib


## Step 1 · What is a Metaflow Card?

A **Metaflow Card** is an HTML report auto-generated at the end of a run (or a specific step).
It lets you share results — tables, plots, metrics — without giving stakeholders access to your code
or requiring them to run anything.

| Mode | Decorator | What you get |
|------|-----------|-------------|
| Default | `@card` | Auto-rendered Markdown from docstrings + artifact table |
| Custom | `@card(type='html')` | Fully custom HTML you build at runtime |
| Component | `@card` + `current.card` | Tables, images, Markdown via the component API |

Cards are stored as run artifacts and viewable with `python flow.py card view [run_id] --origin-pathspec .`
or in the Metaflow UI.

**Key idea:** treat cards like automated PDF reports — a snapshot of what happened, shareable as a URL.


In [ ]:
# Write a flow that uses the default @card decorator
flow_code = '''
from metaflow import FlowSpec, step, card
import random

class DefaultCardFlow(FlowSpec):
    """Demonstrates the default @card — auto-generates an HTML run report."""

    @card   # attach a card to the start step
    @step
    def start(self):
        """Generate some metrics and store them as artifacts."""
        # Any artifact stored here is auto-displayed in the default card
        self.accuracy   = round(random.uniform(0.80, 0.99), 4)
        self.loss       = round(random.uniform(0.01, 0.20), 4)
        self.epochs     = 10
        self.model_name = "LogisticRegression"
        self.next(self.end)

    @step
    def end(self):
        print(f"Run complete. accuracy={self.accuracy}, loss={self.loss}")

if __name__ == "__main__":
    DefaultCardFlow()
'''

with open('/content/default_card_flow.py', 'w') as f:
    f.write(flow_code)
print("default_card_flow.py written")


In [ ]:
# Run the flow — Metaflow stores cards alongside run artifacts
!python /content/default_card_flow.py run


### What just happened?

- `@card` was placed **above** `@step` — decorator order matters; the innermost decorator (`@step`) runs first.
- Metaflow **auto-rendered** all artifacts stored in `start` into a card HTML file.
- The card lives in `.metaflow/DefaultCardFlow/[run_id]/start/card.html` — or retrieve it with `card get`.
- **No code change** is needed to share the report; only the run ID changes between runs.
- In production this card can be published to S3 or served via Metaflow UI with zero extra work.


## Step 2 · Building a Custom Card with Components

The **component API** (`current.card.append(...)`) lets you build rich cards programmatically:

| Component | Import | Description |
|-----------|--------|-------------|
| `Markdown` | `metaflow.cards` | Rendered Markdown text |
| `Table` | `metaflow.cards` | 2-D table from a list-of-lists |
| `Image` | `metaflow.cards` | PNG/JPEG from bytes or file path |
| `Artifact` | `metaflow.cards` | Pretty-print a single artifact |

Components are appended inside the step function **after** computation, so the card always
reflects the final state of that step.

```python
from metaflow.cards import Markdown, Table, Image
from metaflow import current

current.card.append(Markdown("## Results"))
current.card.append(Table([["Metric", "Value"], ["Accuracy", "0.93"]]))
```


In [ ]:
# Flow with a custom card: Table of metrics + a matplotlib plot embedded as Image
custom_card_code = '''
import io
import random
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — required in headless environments
import matplotlib.pyplot as plt

from metaflow import FlowSpec, step, card, current
from metaflow.cards import Markdown, Table, Image

class CustomCardFlow(FlowSpec):
    """Demonstrates custom card components: Markdown, Table, and Image."""

    @card   # enable card for this step
    @step
    def start(self):
        # ── simulate training metrics across 5 epochs ──────────────────
        self.epochs    = list(range(1, 6))
        self.train_acc = [0.60 + i * 0.07 + random.uniform(-0.01, 0.01) for i in range(5)]
        self.val_acc   = [0.58 + i * 0.06 + random.uniform(-0.01, 0.01) for i in range(5)]

        # ── Section 1: Markdown header ──────────────────────────────────
        current.card.append(Markdown("## Training Summary"))
        current.card.append(Markdown("Model: **LogisticRegression** | Dataset: Iris"))

        # ── Section 2: Table of per-epoch metrics ───────────────────────
        rows = [["Epoch", "Train Acc", "Val Acc"]]
        for e, tr, va in zip(self.epochs, self.train_acc, self.val_acc):
            rows.append([str(e), f"{tr:.4f}", f"{va:.4f}"])
        current.card.append(Table(rows))

        # ── Section 3: Matplotlib learning-curve plot as embedded Image ─
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(self.epochs, self.train_acc, marker="o", label="Train")
        ax.plot(self.epochs, self.val_acc,   marker="s", label="Val",   linestyle="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend()
        ax.set_title("Learning Curve")
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format="png", dpi=100)
        buf.seek(0)
        current.card.append(Image(buf.read(), label="Learning Curve"))  # embed PNG bytes
        plt.close(fig)

        self.next(self.end)

    @step
    def end(self):
        print("Custom card generated successfully.")

if __name__ == "__main__":
    CustomCardFlow()
'''

with open('/content/custom_card_flow.py', 'w') as f:
    f.write(custom_card_code)
print("custom_card_flow.py written")


In [ ]:
!python /content/custom_card_flow.py run


### What just happened?

- `current.card.append(...)` **builds the card incrementally** inside the step — every append adds a block.
- `Image(buf.read())` embeds PNG bytes directly into the card HTML as a base-64 `<img>` — **no file server needed**.
- `Table(rows)` accepts a plain Python list-of-lists; the first row becomes the header automatically.
- **`matplotlib.use("Agg")`** is required in any headless environment (Colab, CI, cloud workers) to avoid a display backend error.
- The result is a self-contained HTML file a stakeholder can open with zero dependencies.


## Step 3 · Decorator Review: The Full Stack

After 13 days you have used 8 production-grade decorators. Here is a one-screen reference:

| Decorator | Purpose | Key parameter(s) |
|-----------|---------|------------------|
| `@step` | Mark a method as a DAG step | — |
| `@retry` | Re-run step on failure | `times=3`, `minutes_between_retries=1` |
| `@timeout` | Kill step if it hangs | `seconds=60`, `minutes=5` |
| `@catch` | Intercept exceptions, store in artifact | `var="my_exception"` |
| `@resources` | Request cloud compute (CPU/GPU/RAM) | `cpu=4`, `gpu=1`, `memory=8000` |
| `@conda` | Pin Python env per step | `libraries={"scikit-learn": "1.4.0"}` |
| `@card` | Generate HTML run report | `type='default'` or `type='html'` |
| `@foreach` | Fan-out over a list | first arg to `self.next(self.step, foreach='list')` |

**Stacking rules:**
1. `@step` is always **innermost** (closest to the function).
2. `@card` must be **above** `@step`.
3. All other decorators can be stacked in any order above `@card`.
4. `@retry` + `@timeout` + `@catch` are commonly stacked together on external-facing steps.


In [ ]:
# Demonstrate correct decorator stacking order in a single runnable flow
stacking_code = '''
from metaflow import FlowSpec, step, card, retry, timeout, catch, Parameter

class DecoratorStackFlow(FlowSpec):
    """
    Shows proper decorator stacking:
    @timeout -> @retry -> @catch -> @card -> @step  (outermost to innermost)
    """

    fail_mode = Parameter("fail_mode", default="no",
                          help="Set to 'yes' to trigger a caught exception")

    @card
    @step
    def start(self):
        """Kick off the flow."""
        self.next(self.risky_step)

    @timeout(seconds=30)   # outermost: kill if > 30 s
    @retry(times=1)        # retry once before propagating
    @catch(var="step_err") # catch any remaining exception → store in artifact
    @card                  # generate card even on caught failure
    @step                  # innermost: always last
    def risky_step(self):
        """Step that might fail — all safety decorators applied."""
        if self.fail_mode == "yes":
            raise ValueError("Simulated failure caught by @catch")
        self.result = "success"
        self.next(self.end)

    @step
    def end(self):
        err = getattr(self, "step_err", None)
        if err:
            print(f"Flow completed with caught error: {err}")
        else:
            print(f"Flow completed cleanly. result={self.result}")

if __name__ == "__main__":
    DecoratorStackFlow()
'''

with open('/content/decorator_stack_flow.py', 'w') as f:
    f.write(stacking_code)
print("decorator_stack_flow.py written")


In [ ]:
# Run in normal mode (no failure)
!python /content/decorator_stack_flow.py run --fail_mode no

# Run in fail mode — @catch intercepts the ValueError
!python /content/decorator_stack_flow.py run --fail_mode yes


### What just happened?

- In `--fail_mode no`: the flow ran cleanly through all steps and produced a card for each decorated step.
- In `--fail_mode yes`: `@catch` intercepted the `ValueError`, stored it in `self.step_err`, and the flow **still reached `end`** — no hard crash.
- `@timeout` wraps `@retry` wraps `@catch` — they fire in **outermost-first** order at runtime.
- **The card was generated even on a caught failure** — useful for debugging because you can inspect what artifacts were set before the exception.


## Step 4 · Your Metaflow Mental Model

Writing your mental model in words consolidates scattered knowledge into durable understanding.
Here is a one-page template — revise it with your own phrasing:

---

### Metaflow in one page

**What it is:** A Python framework that turns a class with `@step`-decorated methods into a
versioned, reproducible, cloud-scalable DAG — with zero infrastructure to manage yourself.

**Core loop:**
```
FlowSpec subclass
  └─ @step methods (nodes in DAG)
      └─ self.next(...) wires edges
          └─ foreach / join for fan-out / fan-in
              └─ self.* artifacts auto-persist between steps
```

**Reliability stack (outer → inner):**
```
@timeout  →  @retry  →  @catch  →  @card  →  @step
```

**Resource & env stack:**
```
@resources(cpu, memory, gpu)  →  @conda(libraries)  →  @step
```

**Observability:** `@card` + `current.card.append(...)` → auto-HTML reports per step.

**Data model:** artifacts are snapshots stored in the Metaflow datastore (local S3-like or real S3).
Every run is immutable; you can always replay or inspect past runs with the Client API.

**Why it matters:** you get Git-like versioning for *data and models*, not just code — so ML
experiments are reproducible by default, not by accident.

---

> Take 5 minutes now and rewrite this in your own words in `notes/day-13.md`.


## Step 5 · Listing and Retrieving Cards Programmatically

You can retrieve cards from past runs using the Metaflow Client API — useful for CI pipelines
that automatically publish reports after each run.

The CLI commands you need:

```bash
# List all cards for the latest run of a flow
python flow.py card list

# View a card in your browser (opens HTML)
python flow.py card view --origin-pathspec DefaultCardFlow/latest/start

# Get raw HTML content
python flow.py card get --origin-pathspec DefaultCardFlow/latest/start > report.html
```

In production the Metaflow Service UI renders cards automatically alongside run metadata.


In [ ]:
# Use the Metaflow Client API to inspect the last run programmatically
from metaflow import Flow

try:
    flow = Flow('DefaultCardFlow')
    latest = flow.latest_run
    print(f"Latest run id    : {latest.id}")
    print(f"Successful        : {latest.successful}")
    print(f"Steps in this run : {[step.id for step in latest]}")

    # Retrieve the artifact stored in 'start'
    start_step = latest['start']
    task = list(start_step)[0]   # first (and only) task
    print(f"accuracy artifact : {task.data.accuracy}")
    print(f"model_name        : {task.data.model_name}")
except Exception as e:
    print(f"Note: {e}")
    print("Run the flow cells above first to create a run, then re-execute this cell.")


### What just happened?

- `Flow('DefaultCardFlow')` accesses the **Metaflow datastore** — local by default, S3/Azure in production.
- `flow.latest_run` returns the most recent `Run` object without needing to remember the numeric run ID.
- Iterating over a `Run` yields `Step` objects; iterating over a `Step` yields `Task` objects (one per foreach branch).
- `task.data.accuracy` loads the artifact **lazily** — only fetches the bytes when you access it.
- **This same API works against any historical run** — critical for experiment comparison and audit trails.


## Challenge


In [ ]:
# Challenge: Build a flow called `ModelReportFlow` that:
#   1. Trains a LogisticRegression on the Iris dataset
#   2. Computes accuracy and a per-class classification report
#   3. Generates a @card with:
#      - A Markdown header with the model name and overall accuracy
#      - A Table showing per-class precision / recall / f1
#      - A bar chart (Image) of per-class F1 scores
#   4. Applies @retry(times=2) and @timeout(seconds=60) to the train step

# Scaffold — fill in the blanks:
challenge_code = '''
# --- FILL IN ---
# from metaflow import ...
# from metaflow.cards import ...
# import sklearn, matplotlib, io ...

class ModelReportFlow(FlowSpec):

    @step
    def start(self):
        from sklearn.datasets import load_iris
        data = load_iris(as_frame=True)
        self.X = data.data
        self.y = data.target
        self.target_names = list(data.target_names)
        self.next(self.train)

    # TODO: apply @retry, @timeout, @card here
    @step
    def train(self):
        # TODO: train LogisticRegression, compute classification_report
        # TODO: append Markdown, Table, and Image to current.card
        self.next(self.end)

    @step
    def end(self):
        print("Done!")

if __name__ == "__main__":
    ModelReportFlow()
'''
print(challenge_code)
# Write your solution to /content/model_report_flow.py and run it!


---
## Day 13 key concepts recap

| Concept | What to remember |
|---|---|
| `@card` placement | Always above `@step`; below all other decorators |
| Default card | Auto-renders docstring + artifact table — zero code needed |
| `current.card.append` | Call inside the step; builds card incrementally with Markdown, Table, Image |
| Embed plots | `matplotlib.use('Agg')` + `buf.read()` → `Image(bytes)` |
| Decorator stack order | `@timeout → @retry → @catch → @card → @step` (outer to inner) |
| Client API | `Flow('Name').latest_run['step'][0].data.artifact` — lazy loading |
| Observability purpose | Cards let stakeholders inspect results without touching code or infra |

> **Tip:** Treat flow cards like automated reports — stakeholders can inspect results without
> touching any code.

---
## What's next
**Day 14** → Capstone — build a complete, production-grade end-to-end ML pipeline combining
every decorator and pattern you have learned: ingest, feature engineering, hyperparameter search
with `@foreach`, model registration, and a comprehensive `@card` summary.

Mark Day 13 complete in your [tracker](../index.html).
